# DVS128 Benchmark: EvT-OG vs TimeSformer

This notebook compares the two transformer pipelines currently available in this repo on **DVS128**:

- **EvT-OG**: pretrained checkpoint, inference-only in this setup.
- **TimeSformer**: fine-tuned for **10 epochs** on the reconstructed PNG pipeline.

Questions covered here:

- Which split each model uses.
- Validation vs Test performance.
- Confusion matrices on the test split.
- Validation curves.
- Params / timing / GPU usage where available.

Important limitations:

- In the current EvT-OG code, `val_dataloader()` reads the **official test split**. That means the post-hoc EvT "validation" evaluation is effectively **test** evaluation.
- TimeSformer logs **validation accuracy** but not validation loss in the current training log format.
- Top-5 validation accuracy is available for TimeSformer from the training log, but not for EvT from the saved author training log.

In [ ]:
from __future__ import annotations

import csv
import json
import os
import re
import sys
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from sklearn.metrics import confusion_matrix

ROOT = Path('/home/ppfsa/TESE')
DEV_ROOT = ROOT / 'dev'
EVT_ROOT = DEV_ROOT / 'EvT-OG'
TS_ROOT = DEV_ROOT / 'TimeSformer'

EVT_MODEL_DIR = EVT_ROOT / 'pretrained_models' / 'DVS128_11_24ms_dwn'
EVT_METRICS_CSV = EVT_MODEL_DIR / 'train_log' / 'version_0' / 'metrics.csv'
TS_CFG = TS_ROOT / 'configs' / 'DVS128' / 'TimeSformer_divST_8x1_128_finetune.yaml'
TS_OUTPUT_DIR = TS_ROOT / 'outputs' / 'dvs128_all_samples_finetune'
TS_STDOUT_LOG = TS_OUTPUT_DIR / 'stdout.log'
TS_CHECKPOINT_DIR = TS_OUTPUT_DIR / 'checkpoints'
TS_MANIFEST_DIR = DEV_ROOT / 'datasets' / 'timesformer' / 'DVS128' / 'manifests' / 'all_samples_8f'
GESTURE_MAPPING_CSV = DEV_ROOT / 'datasets' / 'raw' / 'DVS128' / 'gesture_mapping.csv'

DEVICE = 'cuda:0' if torch.cuda.is_available() else 'cpu'

gesture_df = pd.read_csv(GESTURE_MAPPING_CSV)
CLASS_NAMES = gesture_df['action'].tolist()
CLASS_NAMES_0B = [f'{i}. {name.strip()}' for i, name in enumerate(CLASS_NAMES)]

MANUAL_EVT_RUN = {
    'params_m': 0.48,
    'flops_g': 0.08,
    'activated_patches': 47.8,
    'latency_ms': 0.8520,
    'training_val_top1': 96.20,
    'posthoc_test_top1': 96.33,
}

print('Device:', DEVICE)
print('EvT model dir:', EVT_MODEL_DIR)
print('TimeSformer output dir:', TS_OUTPUT_DIR)

## 1. Helpers

The next cell loads the saved training artifacts and exposes two evaluation helpers:

- `evaluate_evt_test()` computes EvT-OG test-side top-1, top-5, confusion matrix, latency, and peak GPU memory.
- `evaluate_timesformer_test()` computes the same for the final TimeSformer checkpoint.

Both are based on the current repo state, not external spreadsheets.

In [ ]:
def parse_json_stats_log(path: Path) -> list[dict]:
    records = []
    for line in path.read_text().splitlines():
        if 'json_stats:' not in line:
            continue
        payload = line.split('json_stats:', 1)[1].strip()
        try:
            records.append(json.loads(payload))
        except json.JSONDecodeError:
            continue
    return records


def load_evt_training_curves() -> pd.DataFrame:
    logs = pd.read_csv(EVT_METRICS_CSV)
    for i, row in logs[logs.epoch.isna()].iterrows():
        candidate = logs[(~logs.epoch.isna()) & (logs.step >= int(row.step))].epoch.min()
        logs.loc[i, 'epoch'] = candidate
    val = logs[~logs['val_acc'].isna()][['epoch', 'val_loss_total', 'val_acc']].copy()
    val['val_top1_acc'] = val['val_acc'] * 100.0
    return val.reset_index(drop=True)


def load_timesformer_curves() -> tuple[pd.DataFrame, pd.DataFrame, dict]:
    records = parse_json_stats_log(TS_STDOUT_LOG)
    train_epoch = pd.DataFrame([r for r in records if r.get('_type') == 'train_epoch'])
    val_epoch = pd.DataFrame([r for r in records if r.get('_type') == 'val_epoch'])
    test_final = next(r for r in records if r.get('split') == 'test_final')
    if not train_epoch.empty:
        train_epoch['epoch_idx'] = train_epoch['epoch'].str.split('/').str[0].astype(int)
        train_epoch['train_top1_acc'] = 100.0 - train_epoch['top1_err'].astype(float)
        train_epoch['train_top5_acc'] = 100.0 - train_epoch['top5_err'].astype(float)
        train_epoch['gpu_mem_gb'] = train_epoch['gpu_mem'].str.rstrip('G').astype(float)
    if not val_epoch.empty:
        val_epoch['epoch_idx'] = val_epoch['epoch'].str.split('/').str[0].astype(int)
        val_epoch['val_top1_acc'] = 100.0 - val_epoch['top1_err'].astype(float)
        val_epoch['val_top5_acc'] = 100.0 - val_epoch['top5_err'].astype(float)
        val_epoch['gpu_mem_gb'] = val_epoch['gpu_mem'].str.rstrip('G').astype(float)
    return train_epoch, val_epoch, test_final


def _push_path(path: Path):
    path_str = str(path)
    if path_str not in sys.path:
        sys.path.insert(0, path_str)


def evaluate_evt_test(device: str = DEVICE) -> dict:
    cwd = Path.cwd()
    os.chdir(EVT_ROOT)
    _push_path(EVT_ROOT)
    try:
        import evaluation_utils as evt_utils
        from trainer import EvNetModel
        from data_generation import Event_DataModule

        all_params = json.loads((EVT_MODEL_DIR / 'all_params.json').read_text())
        path_weights = evt_utils.get_best_weigths(str(EVT_MODEL_DIR), 'val_acc', 'max')
        model = EvNetModel.load_from_checkpoint(
            path_weights,
            map_location=torch.device('cpu'),
            **all_params,
        ).eval().to(device)

        data_params = dict(all_params['data_params'])
        data_params['batch_size'] = 1
        data_params['pin_memory'] = False
        data_params['sample_repetitions'] = 1
        dm = Event_DataModule(**data_params)
        dl = dm.val_dataloader()

        y_true, y_pred, top5_hits = [], [], []
        chunk_times_ms = []
        if torch.cuda.is_available() and device.startswith('cuda'):
            torch.cuda.reset_peak_memory_stats()

        with torch.no_grad():
            for polarity, pixels, labels in dl:
                if polarity is None:
                    continue
                polarity = polarity.to(device)
                pixels = pixels.to(device)
                labels = labels.to(device)
                if torch.cuda.is_available() and device.startswith('cuda'):
                    torch.cuda.synchronize()
                t0 = time.perf_counter()
                _, logits = model(polarity, pixels)
                if torch.cuda.is_available() and device.startswith('cuda'):
                    torch.cuda.synchronize()
                chunk_times_ms.append((time.perf_counter() - t0) * 1000.0 / len(polarity))
                top5 = logits.topk(min(5, logits.shape[1]), dim=1).indices[0].tolist()
                y_true.append(int(labels[0].cpu()))
                y_pred.append(int(logits.argmax(dim=1)[0].cpu()))
                top5_hits.append(int(labels[0].cpu()) in top5)

        cm = confusion_matrix(y_true, y_pred, labels=list(range(len(CLASS_NAMES_0B))), normalize='true')
        peak_gpu_mem_gb = None
        if torch.cuda.is_available() and device.startswith('cuda'):
            peak_gpu_mem_gb = torch.cuda.max_memory_allocated() / (1024 ** 3)

        return {
            'top1_acc': 100.0 * np.mean(np.array(y_true) == np.array(y_pred)),
            'top5_acc': 100.0 * np.mean(top5_hits),
            'confusion_matrix': pd.DataFrame(cm, index=CLASS_NAMES_0B, columns=CLASS_NAMES_0B),
            'latency_ms_per_window': float(np.mean(chunk_times_ms)),
            'peak_gpu_mem_gb': peak_gpu_mem_gb,
            'params_m': sum(p.numel() for p in model.parameters()) / 1e6,
            'split_name': 'official test split via EvT val_dataloader()',
        }
    finally:
        os.chdir(cwd)


def evaluate_timesformer_test(device: str = DEVICE) -> dict:
    cwd = Path.cwd()
    os.chdir(TS_ROOT)
    _push_path(TS_ROOT)
    try:
        from timesformer.config.defaults import get_cfg
        from timesformer.datasets import loader
        from timesformer.models import build_model
        import timesformer.utils.checkpoint as cu

        cfg = get_cfg()
        cfg.merge_from_file(str(TS_CFG))
        cfg.DATA.PATH_TO_DATA_DIR = str(TS_MANIFEST_DIR)
        cfg.DATA.PATH_PREFIX = str(DEV_ROOT / 'datasets' / 'rpg_e2vid')
        cfg.OUTPUT_DIR = str(TS_OUTPUT_DIR)
        cfg.TRAIN.ENABLE = False
        cfg.TEST.ENABLE = True
        cfg.NUM_GPUS = 1 if torch.cuda.is_available() and device.startswith('cuda') else 0
        checkpoint_path = sorted(TS_CHECKPOINT_DIR.glob('checkpoint_epoch_*.pyth'))[-1]
        cfg.TEST.CHECKPOINT_FILE_PATH = str(checkpoint_path)

        model = build_model(cfg)
        cu.load_checkpoint(str(checkpoint_path), model, data_parallel=False)
        test_loader = loader.construct_loader(cfg, 'test')

        preds_all, labels_all, batch_times_ms = [], [], []
        if cfg.NUM_GPUS:
            torch.cuda.reset_peak_memory_stats()

        model.eval()
        with torch.no_grad():
            for inputs, labels, _, _ in test_loader:
                if cfg.NUM_GPUS:
                    inputs = inputs.cuda(non_blocking=True)
                    labels = labels.cuda(non_blocking=True)
                    torch.cuda.synchronize()
                t0 = time.perf_counter()
                preds = model(inputs)
                if cfg.NUM_GPUS:
                    torch.cuda.synchronize()
                batch_times_ms.append((time.perf_counter() - t0) * 1000.0)
                preds_all.append(preds.detach().cpu())
                labels_all.append(labels.detach().cpu())

        preds_all = torch.cat(preds_all)
        labels_all = torch.cat(labels_all)
        top1 = (preds_all.argmax(dim=1) == labels_all).float().mean().item() * 100.0
        top5 = (
            preds_all.topk(min(5, preds_all.shape[1]), dim=1).indices == labels_all.unsqueeze(1)
        ).any(dim=1).float().mean().item() * 100.0
        cm = confusion_matrix(
            labels_all.numpy(),
            preds_all.argmax(dim=1).numpy(),
            labels=list(range(len(CLASS_NAMES_0B))),
            normalize='true',
        )
        peak_gpu_mem_gb = None
        if cfg.NUM_GPUS:
            peak_gpu_mem_gb = torch.cuda.max_memory_allocated() / (1024 ** 3)

        return {
            'top1_acc': top1,
            'top5_acc': top5,
            'confusion_matrix': pd.DataFrame(cm, index=CLASS_NAMES_0B, columns=CLASS_NAMES_0B),
            'latency_ms_per_batch': float(np.mean(batch_times_ms)),
            'latency_ms_per_clip': float(np.mean(batch_times_ms) / cfg.TEST.BATCH_SIZE),
            'peak_gpu_mem_gb': peak_gpu_mem_gb,
            'params_m': sum(p.numel() for p in model.parameters()) / 1e6,
            'checkpoint_path': str(checkpoint_path),
            'split_name': 'official test split',
        }
    finally:
        os.chdir(cwd)


## 2. Load curves and build the benchmark summary

In [ ]:
evt_val_curve = load_evt_training_curves()
ts_train_epoch, ts_val_epoch, ts_test_final = load_timesformer_curves()

evt_eval = evaluate_evt_test()
ts_eval = evaluate_timesformer_test()

summary_rows = [
    {
        'Model': 'EvT-OG',
        'Dataset': 'DVS128',
        'Setup': 'Pretrained inference only',
        'Epochs': int(evt_val_curve['epoch'].max()) + 1,
        'Train split': 'Original author training split',
        'Val split': 'Original author validation log',
        'Test split': evt_eval['split_name'],
        'Val top-1 acc (%)': float(evt_val_curve['val_top1_acc'].max()),
        'Val top-5 acc (%)': np.nan,
        'Test top-1 acc (%)': evt_eval['top1_acc'],
        'Test top-5 acc (%)': evt_eval['top5_acc'],
        'Params (M)': evt_eval['params_m'],
        'FLOPs (G)': MANUAL_EVT_RUN['flops_g'],
        'Latency': f"{evt_eval['latency_ms_per_window']:.3f} ms / window",
        'Peak GPU mem (GB)': evt_eval['peak_gpu_mem_gb'],
        'Notes': 'EvT post-hoc eval uses test split through val_dataloader().',
    },
    {
        'Model': 'TimeSformer',
        'Dataset': 'DVS128',
        'Setup': 'Fine-tuned from SSv2 checkpoint',
        'Epochs': int(ts_train_epoch['epoch_idx'].max()),
        'Train split': 'Official DVS128 train list',
        'Val split': '20% clip split from official train list',
        'Test split': ts_eval['split_name'],
        'Val top-1 acc (%)': float(ts_val_epoch['val_top1_acc'].max()),
        'Val top-5 acc (%)': float(ts_val_epoch['val_top5_acc'].max()),
        'Test top-1 acc (%)': ts_eval['top1_acc'],
        'Test top-5 acc (%)': ts_eval['top5_acc'],
        'Params (M)': ts_eval['params_m'],
        'FLOPs (G)': np.nan,
        'Latency': f"{ts_eval['latency_ms_per_clip']:.3f} ms / clip",
        'Peak GPU mem (GB)': ts_eval['peak_gpu_mem_gb'],
        'Notes': 'Validation loss is not logged by the current TimeSformer run.',
    },
]

summary_df = pd.DataFrame(summary_rows)
summary_df

## 3. Validation curves

- EvT-OG: validation loss and validation top-1 accuracy are available from `metrics.csv`.
- TimeSformer: validation top-1 / top-5 accuracy are available from `stdout.log`, but validation loss is **not** logged in this run.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5), dpi=140)

axes[0].plot(evt_val_curve['epoch'], evt_val_curve['val_loss_total'], label='EvT val loss', color='tab:blue')
axes[0].set_title('EvT-OG Validation Loss')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].grid(alpha=0.3)

axes[1].plot(evt_val_curve['epoch'], evt_val_curve['val_top1_acc'], label='EvT val top-1', color='tab:green')
axes[1].plot(ts_val_epoch['epoch_idx'], ts_val_epoch['val_top1_acc'], label='TimeSformer val top-1', color='tab:orange')
axes[1].plot(ts_val_epoch['epoch_idx'], ts_val_epoch['val_top5_acc'], label='TimeSformer val top-5', color='tab:red', linestyle='--')
axes[1].set_title('Validation Accuracy')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy (%)')
axes[1].grid(alpha=0.3)
axes[1].legend()

plt.tight_layout()
plt.show()

print('TimeSformer validation loss is unavailable in stdout.log for this run.')

## 4. Test confusion matrices

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(18, 7), dpi=140)

for ax, title, cm_df in [
    (axes[0], 'EvT-OG Test Confusion Matrix', evt_eval['confusion_matrix']),
    (axes[1], 'TimeSformer Test Confusion Matrix', ts_eval['confusion_matrix']),
]:
    im = ax.imshow(cm_df.values, cmap='Blues', vmin=0.0, vmax=1.0)
    ax.set_title(title)
    ax.set_xticks(range(len(cm_df.columns)))
    ax.set_yticks(range(len(cm_df.index)))
    ax.set_xticklabels(cm_df.columns, rotation=90, fontsize=8)
    ax.set_yticklabels(cm_df.index, fontsize=8)
    ax.set_xlabel('Predicted')
    ax.set_ylabel('True')

fig.colorbar(im, ax=axes.ravel().tolist(), fraction=0.02, pad=0.02)
plt.tight_layout()
plt.show()

## 5. Resource comparison

In [ ]:
resource_df = summary_df[[
    'Model',
    'Params (M)',
    'FLOPs (G)',
    'Latency',
    'Peak GPU mem (GB)',
    'Val top-1 acc (%)',
    'Test top-1 acc (%)',
    'Test top-5 acc (%)',
]].copy()
resource_df

## 6. What else makes sense to include?

Yes, most of the requested comparison is possible and is now covered here, with two caveats:

- **EvT validation top-5** is not available from the saved author training log.
- **TimeSformer validation loss** is not available from the current run log.

Useful extra sections for a thesis benchmark:

- Per-class recall / per-class error table.
- Macro-F1 in addition to top-1 accuracy.
- Best checkpoint epoch and early-stopping discussion.
- Data-representation discussion: raw event-native tokens vs reconstructed frame clips.
- Split caveats and fairness notes so the benchmark claims stay defensible.